# 11 Quantum Greeks Reconstruction

Estimate Greeks from reconstructed quantum price curves and parameter perturbations, then compare with analytical and finite-difference Greeks.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
spot = 24000.0; K = 24000.0; T = 30 / 365; r = 0.065; sigma = 0.18; option_type = "put"
qg = quantum_greeks_reconstruction(spot, K, T, r, sigma, option_type, n_qubits=int(config["quantum"]["greeks_grid_qubits"]), x_width=float(config["quantum"]["x_width"]))
bs = black_scholes_greeks(spot, K, T, r, sigma, option_type)
fd = finite_difference_greeks(spot, K, T, r, sigma, option_type)
rows = []
for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
    rows.append({"Greek": greek, "Quantum": qg[greek], "AnalyticalBS": bs[greek], "FiniteDifference": fd[greek], "QuantumMinusBS": qg[greek] - bs[greek]})
greek_table = pd.DataFrame(rows)
save_table(greek_table, "11_quantum_greeks_comparison.csv")
for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
    plt.figure()
    vals = greek_table.loc[greek_table["Greek"] == greek, ["Quantum", "AnalyticalBS", "FiniteDifference"]].iloc[0]
    plt.bar(vals.index, vals.values)
    plt.title(f"{greek} comparison at spot")
    plt.ylabel(greek)
    save_current_figure(f"11_{greek.lower()}_comparison.png")
errors = {f"{row.Greek}_abs_error": abs(row.QuantumMinusBS) for row in greek_table.itertuples()}
save_output(errors, "11_quantum_greeks_errors.json")
greek_table
